# 01 — Data Preparation

**Goal:** Build a clean, leakage-free dataset split.

The original Kaggle source has the same images in both its `train/` and `test/` folders,
so we ignore that split entirely. We pool all unique images from `dataset/train/`,
do a **per-class stratified 70 / 15 / 15** split with a fixed seed,
and apply augmentation **only to the training portion**.

Output structure:
```
augmented_dataset/
  train/   ← originals + augmented copies (flip, rotate)
  val/     ← originals only
  test/    ← originals only  (never touched during training)
```

In [2]:
import os
import shutil
import random
from PIL import Image
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)

SOURCE_DIR = "dataset/train"   # sole source — dataset/test is a duplicate
OUTPUT_DIR = "augmented_dataset"

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# remaining 0.15 → test

print(f"Source : {SOURCE_DIR}")
print(f"Output : {OUTPUT_DIR}")
print(f"Split  : {TRAIN_RATIO:.0%} train / {VAL_RATIO:.0%} val / {1-TRAIN_RATIO-VAL_RATIO:.0%} test")

Source : dataset/train
Output : augmented_dataset
Split  : 70% train / 15% val / 15% test


In [3]:
# Augmentation transforms (applied to train only, p=1.0 for offline generation)
flip_transform = transforms.RandomHorizontalFlip(p=1.0)

rotate_contrast_transform = transforms.Compose([
    transforms.RandomRotation(degrees=[-5, 5]),
    transforms.ColorJitter(contrast=0.2)
])

print("Transforms defined.")

Transforms defined.


In [4]:
def build_dataset(source_dir, output_dir, train_ratio, val_ratio):
    """Stratified split + offline augmentation for the train portion."""
    # Clean output dir so re-runs are idempotent
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    classes = sorted(os.listdir(source_dir))
    summary = {}

    for cls in classes:
        cls_dir = os.path.join(source_dir, cls)
        if not os.path.isdir(cls_dir):
            continue

        images = sorted(os.listdir(cls_dir))
        random.shuffle(images)

        n       = len(images)
        n_train = int(train_ratio * n)
        n_val   = int(val_ratio * n)

        splits = {
            'train': images[:n_train],
            'val'  : images[n_train : n_train + n_val],
            'test' : images[n_train + n_val:]
        }

        print(f"\n{cls} ({n} images) → train={len(splits['train'])} "
              f"val={len(splits['val'])} test={len(splits['test'])}")

        aug_counts = {'train': 0, 'val': 0, 'test': 0}

        for split_name, split_images in splits.items():
            out_cls_dir = os.path.join(output_dir, split_name, cls)
            os.makedirs(out_cls_dir, exist_ok=True)

            for img_name in tqdm(split_images, desc=f"{split_name}/{cls}", leave=False):
                img_path = os.path.join(cls_dir, img_name)
                try:
                    img = Image.open(img_path).convert('RGB')
                except Exception:
                    continue

                # Original copy goes to all splits
                img.save(os.path.join(out_cls_dir, f"orig_{img_name}"))
                aug_counts[split_name] += 1

                # Augmentations only for train
                if split_name == 'train':
                    flip_transform(img).save(
                        os.path.join(out_cls_dir, f"flip_{img_name}"))
                    rotate_contrast_transform(img).save(
                        os.path.join(out_cls_dir, f"rot1_{img_name}"))
                    aug_counts[split_name] += 2

                    # Extra augmentations for the minority class
                    if cls == 'ModerateDemented':
                        for i in range(2, 5):
                            rotate_contrast_transform(img).save(
                                os.path.join(out_cls_dir, f"rot{i}_{img_name}"))
                        aug_counts[split_name] += 3

        summary[cls] = aug_counts
        print(f"  Files saved → train={aug_counts['train']} "
              f"val={aug_counts['val']} test={aug_counts['test']}")

    return summary

print("Split function ready.")

Split function ready.


In [5]:
print("Building dataset...")
summary = build_dataset(SOURCE_DIR, OUTPUT_DIR, TRAIN_RATIO, VAL_RATIO)
print("\nDone! Check the 'augmented_dataset' folder.")

Building dataset...

MildDemented (896 images) → train=627 val=134 test=135


train/MildDemented:   0%|          | 0/627 [00:00<?, ?it/s]

val/MildDemented:   0%|          | 0/134 [00:00<?, ?it/s]

test/MildDemented:   0%|          | 0/135 [00:00<?, ?it/s]

  Files saved → train=1881 val=134 test=135

ModerateDemented (64 images) → train=44 val=9 test=11


...

  Files saved → train=6720 val=480 test=480

VeryMildDemented (2240 images) → train=1568 val=336 test=336


train/VeryMildDemented:   0%|          | 0/1568 [00:00<?, ?it/s]

val/VeryMildDemented:   0%|          | 0/336 [00:00<?, ?it/s]

test/VeryMildDemented:   0%|          | 0/336 [00:00<?, ?it/s]

  Files saved → train=4704 val=336 test=336

Done! Check the 'augmented_dataset' folder.


In [6]:
# Verify: no filename overlap between train and test
print("=== Leakage check (filename overlap train vs test) ===")
all_clean = True
for cls in sorted(os.listdir(f"{OUTPUT_DIR}/train")):
    train_files = set(os.listdir(f"{OUTPUT_DIR}/train/{cls}"))
    test_files  = set(os.listdir(f"{OUTPUT_DIR}/test/{cls}"))
    overlap = train_files & test_files
    status  = '❌ LEAKAGE' if overlap else '✅ OK'
    print(f"{cls:20s}: train={len(train_files):5d}  test={len(test_files):5d}  "
          f"overlap={len(overlap)}  {status}")
    if overlap:
        all_clean = False

print()
print("=== Split totals ===")
for split in ['train', 'val', 'test']:
    total = sum(
        len(os.listdir(f"{OUTPUT_DIR}/{split}/{cls}"))
        for cls in os.listdir(f"{OUTPUT_DIR}/{split}")
    )
    print(f"{split:6s}: {total} images")

if all_clean:
    print("\n✅ No data leakage detected. Dataset is ready.")
else:
    print("\n❌ Leakage found — review the split logic.")

=== Leakage check (filename overlap train vs test) ===
MildDemented        : train= 1881  test=  135  overlap=0  ✅ OK
ModerateDemented    : train=  264  test=   11  overlap=0  ✅ OK
NonDemented         : train= 6720  test=  480  overlap=0  ✅ OK
VeryMildDemented    : train= 4704  test=  336  overlap=0  ✅ OK

=== Split totals ===
train : 13569 images
val   : 959 images
test  : 962 images

✅ No data leakage detected. Dataset is ready.
